# Auditoría de reproducibilidad (“Git audit”) · Biolearn
**Artículo:** *A Unified Framework for Systematic Curation and Evaluation of Aging Biomarkers* (preprint en bioRxiv, doi 10.1101/2023.12.02.569722) · **Repositorio:** https://github.com/bio-learn/biolearn

**Qué hace este cuaderno:** instala el código, ejecuta lo que el repositorio dice que funciona, y **anota cada paso como PASS o FAIL con la causa**. Eso es lo que pidió la Dra. Gaur: no copiar las métricas del artículo, sino intentar reproducir y registrar los fallos.

**Cómo usarlo:** Entorno de ejecución → Ejecutar todo (tarda unos 10 a 15 minutos). Si una celda falla, no pasa nada: ese resultado es parte de la auditoría; copia el mensaje.

⚠️ Los datos de NHANES se descargan desde el CDC. Si tu Colab no puede descargarlos, también es un resultado: anótalo.

In [ ]:
import subprocess, sys, os, time, platform
import pandas as pd
WORK = os.getcwd()
LOG = []

def sh(cmd, timeout=1200, cwd=None):
    env = dict(os.environ, MPLBACKEND='Agg')
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, env=env, cwd=cwd or WORK)
    return p.returncode, (p.stdout + p.stderr).strip()

def step(name, cmd, cwd=None, timeout=1200, show=3):
    t = time.time()
    try:
        rc, out = sh(cmd, timeout=timeout, cwd=cwd)
    except subprocess.TimeoutExpired:
        rc, out = -1, 'TIMEOUT'
    status = 'PASS' if rc == 0 else 'FAIL'
    lines = out.splitlines()
    LOG.append({'paso': name, 'estado': status, 'segundos': round(time.time() - t), 'ultima_linea': (lines[-1][:220] if lines else '')})
    print(f'[{status}] {name}')
    for l in lines[-show:]: print('    ', l[:220])
    return rc, out

print('Python      :', sys.version.split()[0])
print('Plataforma  :', platform.platform())
rc, gpu = sh('nvidia-smi -L')
print('GPU         :', gpu if rc == 0 else 'ninguna detectada')
print('Carpeta     :', WORK)

Python      : 3.13.16
Plataforma  : Linux-6.6.122+-x86_64-with-glibc2.39
GPU         : ninguna detectada
Carpeta     : /content


## Paso A · Instalación del lanzamiento publicado (lo que dice el README: `pip install biolearn`)

In [ ]:
step('A1 pip install biolearn lifelines pytest', 'pip install -q biolearn lifelines pytest')
step('A2 version instalada', 'pip show biolearn | grep -i ^version')
for m in ['biolearn.load', 'biolearn.hematology', 'biolearn.data_library', 'biolearn.model']:
    step(f'A3 import {m}', f'python -c "import {m}"')
rc, out = sh('pip show biolearn | grep -i ^requires'); print(out)

[PASS] A1 pip install biolearn lifelines pytest
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 4.9 MB/s eta 0:00:00
        ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 27.1 MB/s eta 0:00:00
        ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 18.9 MB/s eta 0:00:00
[PASS] A2 version instalada
     Version: 0.9.1
[PASS] A3 import biolearn.load
[PASS] A3 import biolearn.hematology
[PASS] A3 import biolearn.data_library
[PASS] A3 import biolearn.model
Requires: appdirs, cvxpy, ecos, matplotlib, numpy, openpyxl, pandas, pyyaml, requests, scikit-learn, scipy, xlrd


## Paso B · El repositorio de GitHub: versión exacta y ejemplos con la versión publicada

In [ ]:
if not os.path.exists('biolearn'):
    step('B1 git clone', 'git clone --depth 1 https://github.com/bio-learn/biolearn.git')
step('B2 commit auditado', "git -C biolearn log -1 --format='%h %ad %s' --date=short")
step('B3 ejemplo NHANES via DataLibrary (version publicada)', 'python biolearn/examples/01_composite_biomarkers/plot_load_nhanes_through_library.py')
step('B4 ejemplo edad fenotipica NHANES (version publicada)', 'python biolearn/examples/01_composite_biomarkers/plot_nhanes.py', timeout=1800)

[PASS] B1 git clone
     Cloning into 'biolearn'...
[PASS] B2 commit auditado
     0d714f5 2026-07-11 test: fix test_dunedin_pace_normalization on pandas 3.0 (fixture orientation) (#210)
[FAIL] B3 ejemplo NHANES via DataLibrary (version publicada)
         data = data_source.load()
                ^^^^^^^^^^^^^^^^
     AttributeError: 'NoneType' object has no attribute 'load'
[PASS] B4 ejemplo edad fenotipica NHANES (version publicada)


(0, '')

## Paso C · Versión actual del repositorio (GitHub) y sus pruebas

In [ ]:
step('C1 pip install del clon', 'pip install -q ./biolearn')
step('C2 version tras instalar el clon', 'pip show biolearn | grep -i ^version')
step('C3 pruebas del repositorio (pytest)', 'python -m pytest tests -q', cwd=os.path.join(WORK, 'biolearn'))
step('C4 ejemplo NHANES via DataLibrary (version GitHub)', 'python biolearn/examples/01_composite_biomarkers/plot_load_nhanes_through_library.py')
step('C5 ejemplo edad fenotipica NHANES (version GitHub)', 'python biolearn/examples/01_composite_biomarkers/plot_nhanes.py', timeout=1800)

[PASS] C1 pip install del clon
[PASS] C2 version tras instalar el clon
     Version: 0.1.dev1+g0d714f5a0
[PASS] C3 pruebas del repositorio (pytest)
     ....                                                                     [100%]
     4 passed in 15.25s
[PASS] C4 ejemplo NHANES via DataLibrary (version GitHub)
       warnings.warn(
     /usr/local/lib/python3.13/dist-packages/biolearn/clinical/convert.py:94: UserWarning: Biomarker 'mean_cell_hemoglobin': 1 values outside expected range [15.0, 45.0] (unit: pg). Check units.
       warnings.warn(
[PASS] C5 ejemplo edad fenotipica NHANES (version GitHub)


(0, '')

## Paso D · ¿Da el mismo número dos veces? (determinismo)
Se calcula la edad fenotípica en NHANES 2010 en dos procesos distintos y se compara una huella del resultado.

In [ ]:
CHECK = '''
import hashlib, numpy as np
from biolearn.load import load_nhanes
from biolearn.hematology import phenotypic_age
df = load_nhanes(2010)
pa = phenotypic_age(df)
print(len(df), hashlib.sha256(np.round(pa.values.astype(float), 6).tobytes()).hexdigest()[:16])
'''
open('check_det.py', 'w').write(CHECK)
r1 = step('D1 corrida 1', 'python check_det.py', show=1)
r2 = step('D2 corrida 2', 'python check_det.py', show=1)
same = (r1[0] == 0 and r2[0] == 0 and r1[1].splitlines()[-1] == r2[1].splitlines()[-1])
print('Resultado identico en las dos corridas:', same)
LOG.append({'paso': 'D3 identico en 2 corridas', 'estado': 'PASS' if same else 'FAIL', 'segundos': 0, 'ultima_linea': ''})

[PASS] D1 corrida 1
     2877 99cba9ed4956fe90
[PASS] D2 corrida 2
     2877 99cba9ed4956fe90
Resultado identico en las dos corridas: True


## Registro de la auditoría

In [ ]:
log = pd.DataFrame(LOG)
pd.set_option('display.max_colwidth', 120)
display(log)
log.to_csv('audit_log.csv', index=False)
print('Guardado: audit_log.csv')
rc, out = sh('pip freeze | grep -i -E "^(numpy|pandas|scipy|scikit-learn|biolearn|lifelines|torch|matplotlib)=="')
print(out)

,paso,estado,segundos,ultima_linea
0,A1 pip install biolearn lifelines pytest,PASS,31,━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 18.9 MB/s eta 0:00:00
1,A2 version instalada,PASS,2,Version: 0.9.1
2,A3 import biolearn.load,PASS,1,
3,A3 import biolearn.hematology,PASS,1,
4,A3 import biolearn.data_library,PASS,3,
5,A3 import biolearn.model,PASS,8,
6,B1 git clone,PASS,3,Cloning into 'biolearn'...
7,B2 commit auditado,PASS,0,0d714f5 2026-07-11 test: fix test_dunedin_pace_normalization on pandas 3.0 (fixture orientation) (#210)
8,B3 ejemplo NHANES via DataLibrary (version publicada),FAIL,16,AttributeError: 'NoneType' object has no attribute 'load'
9,B4 ejemplo edad fenotipica NHANES (version publicada),PASS,28,


Guardado: audit_log.csv
lifelines==0.30.3
matplotlib==3.10.0
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
scipy==1.16.3


## Lista de reproducibilidad (Yes / Partial / No) — Biolearn, preprint v6 (posted May 31, 2024)
| # | Item | Score | Evidence |
|---|---|---|---|
| 1 | Seeds and repeated runs | **No** | States reproducibility "was ensured" but gives no seeds or repeats. |
| 2 | Splits described | **Partial** | "Randomized cross-validation" without folds or repeats; evaluation datasets are named with N. |
| 3 | Leakage checked | **No** | No leakage check described. |
| 4 | Separate fitting | **No** | Does not say whether clock training data overlap with evaluation cohorts. |
| 5 | Statistical test named | **Partial** | Pearson and Cox (two-sided P) named; no multiple-testing correction (searched "Bonferroni", "multiple testing", "multiple comparison", "false discovery": 0 hits). |
| 6 | Effect size and interval | **Yes (with reserve)** | Hazard ratios per SD with 95% CI in Cox figures; bootstrap mentioned "where applicable" without saying where. |
| 7 | External validation | **Partial** | Several cohorts (NAS N=1,488; JenAge N=62; NHANES 2010 N=2,877) but overlap with clock training data not addressed. |
| 8 | Compute documented | **Partial** | Python >=3.10; no pinned versions or hardware; PyPI 0.9.1 and GitHub master behave differently (step B3 FAIL). |
| 9 | Code and data | **Yes** | Code on GitHub (BSD license stated in the repository); public data. The preprint's CC-BY-ND notice covers the text only. |


## Audit report
**1. Paper metadata.** *A Unified Framework for Systematic Curation and Evaluation of Aging Biomarkers* (Ying et al.), bioRxiv preprint, doi 10.1101/2023.12.02.569722, version posted May 31, 2024 (v6). Later published in *Nature Aging* 5(11), 2025. Repository: github.com/bio-learn/biolearn. Audited commit: `0d714f5` (2026-07-11).

**2. Experimental setup.** The preprint evaluates 17 epigenetic clocks on the Normative Aging Study (N=1,488; 38.8% deceased) with Cox models adjusted for age and sex, transcriptomic clocks on JenAge (N=62), and Phenotypic Age on NHANES 2010 (N=2,877), among other datasets.

**3. Leakage check.** The paper reports no leakage check and does not state whether the clocks' training data overlap with the evaluation cohorts.

**4. Statistical rigor.** It states that robustness and reproducibility "were ensured" by randomized cross-validation and bootstrapping "where applicable", but gives no folds, seeds, repeats, or location of the bootstrap. Cox hazard ratios per SD with 95% CIs and two-sided P-values are reported. No multiple-testing correction was found. The text mentions 17 clocks and six event categories, whereas Fig. 3c shows 14 clocks and a later sentence refers to five outcomes.

**5. Execution of the repository (my own run, Google Colab).** Installing the PyPI release (v0.9.1) and importing four modules worked. The NHANES example failed on PyPI with `AttributeError: 'NoneType' object has no attribute 'load'` but passed on GitHub master, so the published package and the repository disagree. The repository's 4 tests passed. Two independent runs of `phenotypic_age` on NHANES 2010 gave identical output (2,877 rows, hash 99cba9ed4956fe90). In a separate clean install (outside Colab) I observed that `biolearn.model` imports `torch`, which is not declared as a dependency.

**6. Missing information.** Seeds, cross-validation design, pinned dependency versions, hardware, statement on overlap between clock training data and evaluation cohorts, and a software license / data-deposition statement in the paper (the preprint's CC-BY-ND notice covers the text only; the BSD license is stated in the repository).

**7. Overall score and recommendations.** Partially reproducible: the code is deterministic and the tests pass, but the paper omits validation details and the installable release does not match the repository. Top 3 fixes: 1) publish a PyPI release matching master, or fix the README install instructions; 2) pin dependencies and declare `torch`; 3) report the cross-validation design, seeds, leakage checks, bootstrap CIs and a multiple-testing correction.


In [ ]:
from IPython.display import Markdown, display

n_pass = (log['estado'] == 'PASS').sum()
n_fail = (log['estado'] == 'FAIL').sum()

display(Markdown(f"""
## Ticket de salida — Aly Gallo

**Resumen de mi ejecución (Colab):** {len(log)} pasos, **{n_pass} PASS** y **{n_fail} FAIL**.
El fallo ({', '.join(log.loc[log['estado']=='FAIL', 'paso'])}) ocurre con la versión publicada en PyPI (0.9.1) y no con la de GitHub.

**Antes pensaba** que si un paper publica su código en GitHub, cualquiera puede reproducirlo; **ahora pienso** que hay que ejecutarlo, porque la versión instalable puede fallar aunque el repositorio funcione.
"""))


## Ticket de salida — Aly Gallo

**Resumen de mi ejecución (Colab):** 18 pasos, **17 PASS** y **1 FAIL**.
El fallo (B3 ejemplo NHANES via DataLibrary (version publicada)) ocurre con la versión publicada en PyPI (0.9.1) y no con la de GitHub.

**Antes pensaba** que si un paper publica su código en GitHub, cualquiera puede reproducirlo; **ahora pienso** que hay que ejecutarlo, porque la versión instalable puede fallar aunque el repositorio funcione.
